## Baseline Invoice extraction using GPT5 and MLflow 3.x

In [0]:
# ! pip install -q openai datasets pandas tqdm dotenv mlflow

### Imports

In [0]:
from datasets import load_dataset
from openai import OpenAI
import os
import json
import time
import mlflow
from utils import (
    generate_urls,
    calculate_invoice_accuracies,
    calculate_key_level_metrics,
    calculate_individual_invoice_accuracies,
    convert_base64_to_pil,
    retrieve_token_usage,
)

from prompt import register_prompt

from dotenv import load_dotenv
from mlflow.entities import Feedback
from mlflow.genai import scorer

load_dotenv()

### Config

In [0]:
MLFLOW_TRACKING_URI = "http://localhost:5000/"
MODEL_NAME = "gpt-5-nano"
REASONING = "low"
MLFLOW_EXPERIMENT_NAME = "cord-v2-gpt5-baseline"
PROMPT_NAME = "invoice-extraction-gpt5-prompt"
PROMPT_VERSION = "1"

### Initialize MLflow and OpenAI environment

In [0]:
client = OpenAI()
os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
os.environ["MLFLOW_EXPERIMENT_NAME"] = MLFLOW_EXPERIMENT_NAME
mlflow.openai.autolog()

### Register prompt and model

In [0]:
register_prompt(prompt_name=PROMPT_NAME)

### Load the dataset

In [0]:
dataset = load_dataset("naver-clova-ix/cord-v2")
dataset

In [0]:
dataset["test"][0]["image"]

In [0]:
print(dataset["test"][0]['ground_truth'])

In [0]:
example_1 = json.loads(dataset["validation"][0]["ground_truth"])["gt_parse"]
example_2 = json.loads(dataset["validation"][1]["ground_truth"])["gt_parse"]
example_3 = json.loads(dataset["validation"][2]["ground_truth"])["gt_parse"]

### Data Preparation

In [0]:
with open("schema.json", "r") as f:
    schema_dict = json.load(f)

schema_dict

In [0]:
NUM_SAMPLES = 10
test_dataset = dataset["test"].select(range(NUM_SAMPLES))
url_list, ground_truth_list = generate_urls(dataset=test_dataset)

### Inference and Evaluation

In [0]:
import os
if not os.path.exists("artifacts"):
    os.makedirs("artifacts")

eval_dataset = []
for index, url in enumerate(url_list):
    eval_dict = {
        "inputs": {"image_base64": url, "schema": schema_dict},
        "expectations": {"expected_response": ground_truth_list[index]},
    }
    eval_dataset.append(eval_dict)

eval_dataset

In [0]:
def predict_fn(image_base64, schema) -> str:
    system_prompt_template = mlflow.genai.load_prompt(
        name_or_uri=PROMPT_NAME, version=PROMPT_VERSION
    )
    if "fewshot" in PROMPT_NAME:
        system_prompt = system_prompt_template.format(
            schema=schema, example1=example_1, example2=example_2, example3=example_3
        )
    else:
        system_prompt = system_prompt_template.format(schema=schema)

    response = client.responses.create(
        model=MODEL_NAME,
        reasoning={
            "effort": REASONING,
        },
        text={"verbosity": "low"},
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": system_prompt},
                    {"type": "input_image", "image_url": f"data:image/jpeg;base64,{image_base64}"},
                ],
            }
        ],
    )
    response_text = response.output[1].content[0].text

    return response_text

In [0]:
@scorer
def exact_match(inputs, outputs, expectations, trace) -> Feedback:
    outputs = json.loads(outputs)
    expectations = expectations["expected_response"]
    trace_id = trace.info.trace_id

    # Create child run for every invoice
    with mlflow.start_run(parent_run_id=parent_run.info.run_id, nested=True, run_name=trace_id):
        # Log the prediction and ground truth
        mlflow.log_dict(outputs, "prediction.json")
        mlflow.log_dict(expectations, "ground_truth.json")

        # Compute accuracy @ invoice level
        pred_df, acc = calculate_individual_invoice_accuracies(
            ground_truth=expectations, output=outputs
        )
        mlflow.log_param("trace_id", trace_id)
        mlflow.log_metric("accuracy", acc)

        # Log predictions as artifacts
        pred_df.to_csv(f"artifacts/predictions_{trace_id}.csv", index=False)
        mlflow.log_artifact(local_path=f"artifacts/predictions_{trace_id}.csv")

        # Save the invoice image
        base64_string = inputs["image_base64"]
        image = convert_base64_to_pil(base64_string)
        mlflow.log_image(image, f"input_image_{trace_id}.png")

    return Feedback(value=round(acc, 2), name="Accuracy")

In [0]:

parent_run = mlflow.start_run(run_name=f"{MODEL_NAME}-{REASONING}-evaluation")

start_time = time.time()

results = mlflow.genai.evaluate(
    data=eval_dataset,
    scorers=[exact_match],
    predict_fn=predict_fn,
)

total_time = time.time() - start_time


In [0]:
total_time = time.time() - start_time
# Log model details
mlflow.log_params(
    {
        "model_name": MODEL_NAME,
    }
)
mlflow.log_param("reasoning", REASONING)

mlflow.log_param("prompt", PROMPT_NAME)

# Log total number of input and output tokens - to estimate the overall cost
trace_df = mlflow.search_traces(run_id=parent_run.info.run_id)
input_tokens, output_tokens, reasoning_tokens = retrieve_token_usage(trace_df)
mlflow.log_metric("total_input_tokens", input_tokens)
mlflow.log_metric("total_output_tokens", output_tokens)
mlflow.log_metric("total_reasoning_tokens", reasoning_tokens)

# Log cost estimation
with open("cost.json", "r") as f:
    cost_dict = json.load(f)

total_input_cost = (cost_dict[MODEL_NAME]["input"] * input_tokens) / 10**6
total_output_cost = (cost_dict[MODEL_NAME]["output"] * output_tokens) / 10**6
mlflow.log_metric("estimated_cost", total_input_cost + total_output_cost)

# Log aggregated invoice metrics
response_str_list = trace_df["response"].tolist()
response_list = []
for response_str in response_str_list:
    try:
        response_list.append(json.loads(response_str["output"][1]["content"][0]["text"]))
    except Exception:
        response_list.append("")
calculate_invoice_accuracies(response_list, ground_truth_list)
mlflow.log_artifact("artifacts/invoice_metrics.csv")


# Log aggregated key level metrics
key_level_metrics_df = calculate_key_level_metrics(response_list, ground_truth_list)
mlflow.log_artifact("artifacts/key_metrics.csv")

# Log total execution time
mlflow.log_metric("total_execution_time", total_time)

# End the parent run
mlflow.end_run()